# Surya pipeline validation

Part A: run the official `easy_inference` quickstart on a tiny date window to confirm the environment, weight download, SDO data download, and forward pass all work on a Colab GPU.

Part B: override `finetune=True` on the loaded model to get raw embedding tokens (pre-decoder) instead of the pixel-space forecast — this is the actual object this project needs. Best-effort: not test-run against a live GPU before landing here, debug inline if a shape/import breaks.

Runtime: Colab, GPU (Runtime > Change runtime type > T4 GPU).

In [ ]:
!git clone https://github.com/NASA-IMPACT/Surya.git
%cd Surya
!pip install -q -e .
!pip install -q h5netcdf huggingface_hub

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no GPU -- set Runtime > Change runtime type > T4 GPU')

## Part A: official quickstart, tiny window

Edits `config_easy.yaml` to a 1-hour window (smallest useful test), non-interactive, single rollout step.

In [ ]:
import yaml

cfg_path = 'easy_inference/config_easy.yaml'
with open(cfg_path) as f:
    cfg = yaml.safe_load(f)

cfg['user']['start_datetime'] = '2014-10-23 10:00:00'
cfg['user']['end_datetime'] = '2014-10-23 11:00:00'
cfg['user']['prompt_for_dates'] = False
cfg['user']['rollout_steps'] = 1
cfg['user']['output_dir'] = 'outputs_pipeline_check'

with open(cfg_path, 'w') as f:
    yaml.safe_dump(cfg, f)

print(yaml.safe_dump(cfg))

In [ ]:
!python easy_inference/run_easy_inference.py --config-path easy_inference/config_easy.yaml

If this produced `outputs_pipeline_check/prediction.nc` without errors, the full pipeline (weights, data download, GPU forward pass) is confirmed working end to end.

In [ ]:
import os
for root, _, files in os.walk('outputs_pipeline_check'):
    for f in files:
        print(os.path.join(root, f))

## Part B: extract embedding tokens instead of the decoded forecast

`HelioSpectFormer.forward()` returns raw backbone tokens when `self.finetune=True`, skipping the pixel-space decoder entirely (see `surya/models/helio_spectformer.py`, the `if self.finetune: return tokens` branch right after `tokens = self.backbone(tokens)`). Reusing the exact model/config the quickstart just built and downloaded, toggling that flag after construction, and running a normal batch through it should hand back the embedding directly — no need to reimplement Surya's own data loading.

In [ ]:
import sys
sys.path.insert(0, 'easy_inference')

import run_easy_inference as ezi
import inspect

# Sanity check: confirm build_model and the dataset/dataloader entrypoints exist
# under these names before relying on them (script internals may differ by version).
print([n for n in dir(ezi) if 'build' in n.lower() or 'dataset' in n.lower() or 'loader' in n.lower()])

In [ ]:
# Rebuild the same base_config the quickstart used, then flip finetune on the model.
# If `run_easy_inference` exposes its config-loading + dataloader-building steps under
# different names than guessed here, inspect `dir(ezi)` above and adjust the calls below
# rather than guessing blind -- this cell is the one most likely to need a live fix.

import torch

with open(cfg_path) as f:
    full_cfg = yaml.safe_load(f)

# base_config assembly mirrors what run_easy_inference.main() does with user+advanced
# sections before calling build_model(base_config); adjust if main() does more.
advanced = full_cfg['advanced']
with open(advanced['foundation_config_path']) as f:
    base_config = yaml.safe_load(f)

base_config['data']['time_delta_input_minutes'] = advanced['time_delta_input_minutes']
base_config['data']['n_input_timestamps'] = len(advanced['time_delta_input_minutes'])

model = ezi.build_model(base_config)
state = torch.load(advanced['weights_path'], map_location='cpu')
model.load_state_dict(state if not isinstance(state, dict) or 'state_dict' not in state else state['state_dict'])
model.eval()

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)

model.finetune = True  # forward() now returns backbone tokens, not the decoded forecast
print('finetune flag set:', model.finetune)

In [ ]:
# Build one real batch the same way the quickstart's dataloader does, reusing its
# dataset class directly rather than hand-rolling normalization/channel order.

index_path = advanced['index_path']
print('index file exists:', os.path.exists(index_path))

# Look for the Dataset class run_easy_inference.py instantiates around its dataloader
# construction (grep showed `DataLoader(**kwargs)` plus an `InputOnlyRolloutDataset`
# reference) -- confirm the class name matches before use.
print([n for n in dir(ezi) if 'Dataset' in n])

In [ ]:
# Once the dataset class name is confirmed from the previous cell's output, build a
# single-sample batch and run it through the model with finetune=True.
#
# dataset = ezi.InputOnlyRolloutDataset(index_path=index_path, ...)  # fill in real args
# batch = next(iter(torch.utils.data.DataLoader(dataset, batch_size=1)))
# batch = {k: v.to(device) for k, v in batch.items() if torch.is_tensor(v)}
#
# with torch.no_grad():
#     tokens = model(batch)
# print('embedding tokens shape:', tokens.shape)
# import numpy as np
# np.save('sample_embedding.npy', tokens.cpu().numpy())

print('Fill in dataset construction using the class name printed above, then uncomment.')